In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('Brazilian E-Commerce Public Dataset by Olist.csv')


customers = df[['customer_id', 'customer_unique_id', 'customer_state']].drop_duplicates()
orders = df[['order_id', 'customer_id', 'order_purchase_timestamp']].drop_duplicates()
payments = df[['order_id', 'payment_value']].drop_duplicates()

print("Data loaded successfully!")

Data loaded successfully!


In [ ]:

conn = sqlite3.connect(':memory:')
customers.to_sql('Customers', conn, index=False)
orders.to_sql('Orders', conn, index=False)
payments.to_sql('Payments', conn, index=False)

query = """
WITH OrderMonths AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month
    FROM Orders o
    JOIN Customers c ON o.customer_id = c.customer_id
),
CohortMonths AS (
    SELECT 
        customer_unique_id,
        MIN(order_month) AS cohort_month
    FROM OrderMonths
    GROUP BY customer_unique_id
)
SELECT 
    o.customer_unique_id,
    c.cohort_month,
    o.order_month,
    p.payment_value
FROM OrderMonths o
JOIN CohortMonths c ON o.customer_unique_id = c.customer_unique_id
JOIN Payments p ON o.order_id = p.order_id
"""
base_data = pd.read_sql_query(query, conn)

In [ ]:
base_data['cohort_month'] = pd.to_datetime(base_data['cohort_month'])
base_data['order_month'] = pd.to_datetime(base_data['order_month'])


years_diff = base_data['order_month'].dt.year - base_data['cohort_month'].dt.year
months_diff = base_data['order_month'].dt.month - base_data['cohort_month'].dt.month
base_data['cohort_index'] = years_diff * 12 + months_diff + 1


cohort_counts = base_data.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().reset_index()


cohort_pivot = cohort_counts.pivot(index='cohort_month', columns='cohort_index', values='customer_unique_id')


cohort_sizes = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0)


retention_matrix.to_csv('retention_heatmap_data.csv')